# Splitting Dataset

In [18]:
dataset_path = '/content/drive/MyDrive/UTY/Semester 7/Pemrosesan Teks /Labeling Data/Raw/labeled_data_hok_250.csv'

In [19]:
import pandas as pd

# Load the dataset
df = pd.read_csv(dataset_path)

# Assuming a column named 'label' exists to determine if data is labeled or not.
# If your label column has a different name, please change 'label_column_name' below.
# For example, if your label column is named 'sentiment', change it to 'sentiment'.
label_column_name = 'label'

# Separate labeled and non-labeled data
# Labeled data: rows where the 'label_column_name' is not null
# Non-labeled data: rows where the 'label_column_name' is null

labeled_data = df[df[label_column_name].notna()]
non_labeled_data = df[df[label_column_name].isna()]

# Save the separated data into new CSV files
labeled_data.to_csv('/content/drive/MyDrive/UTY/Semester 7/Pemrosesan Teks /Labeling Data/labeled_hok.csv', index=False)
non_labeled_data.to_csv('/content/drive/MyDrive/UTY/Semester 7/Pemrosesan Teks /Labeling Data/non_labeled_hok.csv', index=False)

print(f"Labeled data saved to labeled.csv ({len(labeled_data)} rows)")
print(f"Non-labeled data saved to non_labeled.csv ({len(non_labeled_data)} rows)")
print("Please note: This code assumes a column named 'label' exists to distinguish labeled from non-labeled data. If your dataset uses a different column name for labels, please modify the 'label_column_name' variable accordingly.")


Labeled data saved to labeled.csv (250 rows)
Non-labeled data saved to non_labeled.csv (750 rows)
Please note: This code assumes a column named 'label' exists to distinguish labeled from non-labeled data. If your dataset uses a different column name for labels, please modify the 'label_column_name' variable accordingly.


# Modeling

## Text Preprocessing and Feature Extraction


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000, # Consider top 5000 features based on term frequency
    min_df=5,          # Ignore words that appear in less than 5 documents
    max_df=0.7,        # Ignore words that appear in more than 70% of the documents
    stop_words='english' # Remove common English stop words
)

# Fit the TfidfVectorizer on the 'content' column of the entire dataset (df)
tfidf_vectorizer.fit(df['content'])

# Transform the 'content' column of labeled_data
X_labeled = tfidf_vectorizer.transform(labeled_data['content'])

# Transform the 'content' column of non_labeled_data
X_non_labeled = tfidf_vectorizer.transform(non_labeled_data['content'])

print("TF-IDF vectorization completed.")
print(f"Shape of X_labeled: {X_labeled.shape}")
print(f"Shape of X_non_labeled: {X_non_labeled.shape}")

TF-IDF vectorization completed.
Shape of X_labeled: (250, 458)
Shape of X_non_labeled: (750, 458)


## Train Initial Classifier



In [21]:
from sklearn.linear_model import LogisticRegression

# Prepare the target variable y_labeled from the 'label' column of labeled_data
y_labeled = labeled_data['label']

# Instantiate the Logistic Regression classifier
classifier = LogisticRegression(max_iter=1000) # Increased max_iter for convergence

# Train the classifier using X_labeled (features) and y_labeled (target)
classifier.fit(X_labeled, y_labeled)

print("Logistic Regression classifier trained successfully.")

Logistic Regression classifier trained successfully.


## Predict Pseudo-Labels on Non-Labeled Data



In [22]:
import numpy as np

# Predict probability scores for the non-labeled data
pseudo_label_probabilities = classifier.predict_proba(X_non_labeled)

# Predict labels for the non-labeled data
pseudo_labels = classifier.predict(X_non_labeled)

print("Pseudo-label probabilities predicted successfully.")
print(f"Shape of pseudo_label_probabilities: {pseudo_label_probabilities.shape}")
print("Pseudo-labels predicted successfully.")
print(f"Shape of pseudo_labels: {pseudo_labels.shape}")

# Display the first 5 pseudo-label probabilities and predicted labels
print("\nFirst 5 pseudo-label probabilities:\n", pseudo_label_probabilities[:5])
print("\nFirst 5 pseudo-labels:\n", pseudo_labels[:5])

Pseudo-label probabilities predicted successfully.
Shape of pseudo_label_probabilities: (750, 3)
Pseudo-labels predicted successfully.
Shape of pseudo_labels: (750,)

First 5 pseudo-label probabilities:
 [[0.34540576 0.21718972 0.43740452]
 [0.3707476  0.23986189 0.38939052]
 [0.3707476  0.23986189 0.38939052]
 [0.3707476  0.23986189 0.38939052]
 [0.31500196 0.36529042 0.31970762]]

First 5 pseudo-labels:
 ['positive' 'positive' 'positive' 'positive' 'neutral']


## Select High-Confidence Pseudo-Labels

In [23]:
confidence_threshold = 0.9

# Get the maximum probability for each pseudo-label prediction
max_probabilities = np.max(pseudo_label_probabilities, axis=1)

# Create a mask for high-confidence predictions
high_confidence_mask = max_probabilities >= confidence_threshold

# Filter X_non_labeled and pseudo_labels based on the high-confidence mask
X_high_confidence = X_non_labeled[high_confidence_mask]
y_high_confidence = pseudo_labels[high_confidence_mask]

print(f"Defined confidence threshold: {confidence_threshold}")
print(f"Number of high-confidence pseudo-labels: {len(y_high_confidence)}")
print(f"Shape of X_high_confidence: {X_high_confidence.shape}")
print(f"Shape of y_high_confidence: {y_high_confidence.shape}")

# Display the first few high-confidence pseudo-labels and their max probabilities
print("\nFirst 5 high-confidence pseudo-labels:", y_high_confidence[:5])
print("First 5 corresponding max probabilities:", max_probabilities[high_confidence_mask][:5])

Defined confidence threshold: 0.9
Number of high-confidence pseudo-labels: 26
Shape of X_high_confidence: (26, 458)
Shape of y_high_confidence: (26,)

First 5 high-confidence pseudo-labels: ['positive' 'positive' 'positive' 'positive' 'positive']
First 5 corresponding max probabilities: [0.91314769 0.91314769 0.91314769 0.91314769 0.93616064]


## Combine Pseudo-Labeled Data with Labeled Data


In [24]:
import scipy.sparse as sp

# Concatenate original labeled features and high-confidence pseudo-labeled features
X_combined = sp.vstack([X_labeled, X_high_confidence])

# Convert y_labeled to a numpy array for consistent concatenation
y_labeled_np = y_labeled.to_numpy()

# Concatenate original labeled target variable and high-confidence pseudo-labels
y_combined = np.concatenate([y_labeled_np, y_high_confidence])

print("Combined data created successfully.")
print(f"Shape of X_combined: {X_combined.shape}")
print(f"Shape of y_combined: {y_combined.shape}")

Combined data created successfully.
Shape of X_combined: (276, 458)
Shape of y_combined: (276,)


## Retrain Model with Augmented Data


In [25]:
from sklearn.linear_model import LogisticRegression

# Initialize a new Logistic Regression classifier
retrained_classifier = LogisticRegression(max_iter=1000)

# Train the classifier using the combined dataset
retrained_classifier.fit(X_combined, y_combined)

print("Retrained Logistic Regression classifier successfully using combined data.")

Retrained Logistic Regression classifier successfully using combined data.


## Saving Result

In [26]:
import pandas as pd

# Predict labels for the original non_labeled_data using the retrained_classifier
predictions_self_trained = retrained_classifier.predict(X_non_labeled)

# Create a copy of the non_labeled_data to add predictions
self_trained_data = non_labeled_data.copy()

# Add the predicted labels to the non-labeled data DataFrame
self_trained_data['self_trained_label'] = predictions_self_trained

# Save the DataFrame with self-trained labels to a new CSV file
output_path = '/content/drive/MyDrive/UTY/Semester 7/Pemrosesan Teks /Labeling Data/self_trained_hok.csv'
self_trained_data.to_csv(output_path, index=False)